In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
chexpert = pd.read_csv('mimic-cxr-2.0.0-chexpert.csv.gz')
metadata = pd.read_csv('mimic-cxr-2.0.0-metadata.csv.gz')
split = pd.read_csv('mimic-cxr-2.0.0-split.csv.gz')

In [ ]:
# MIMIC-CXR-JPG CSV 데이터 구조 확인
# - CheXpert, Metadata, Split 데이터의 행·열 크기 확인
# - 각 데이터의 컬럼명 및 상위 5개 행 확인
# - 데이터 구성과 주요 변수 파악

for name, df in [
    ("CheXpert", chexpert),
    ("Metadata", metadata),
    ("Split", split)
]:
    print(f"\n===== {name} =====")
    print("데이터 크기:", df.shape)
    print("컬럼 목록:", df.columns.tolist())
    display(df.head())

In [ ]:
# ============================================================
# X-ray 촬영 방향(ViewPosition) 분포 분석
# - PA, AP, LATERAL 등 촬영 방향별 이미지 수 확인
# - 촬영 방향 정보의 결측값 확인
# ============================================================

metadata["ViewPosition"].value_counts(dropna=False)

In [ ]:
# ============================================================
# Train / Validate / Test 데이터 분포 확인
# - 데이터 분할별 X-ray 이미지 수 계산
# - 전체 이미지 대비 비율(%) 확인
# ============================================================

split_distribution = pd.DataFrame({
    "Count": split["split"].value_counts(),
    "Percent": (split["split"].value_counts(normalize=True) * 100).round(2)
})

display(split_distribution)

In [ ]:
print("Train:", (split['split'] == 'train').sum())
print("Validate:", (split['split'] == 'validate').sum())
print("Test:", (split['split'] == 'test').sum())

In [ ]:
# MIMIC-CXR-JPG 데이터별 고유 식별자 수 확인
# - subject_id: 고유 환자 수
# - study_id: 고유 검사 수
# - dicom_id: 고유 X-ray 이미지 수
# - CheXpert, Metadata, Split 간 데이터 규모 비교
for name, df in [
    ("CheXpert", chexpert),
    ("Metadata", metadata),
    ("Split", split)
]:
    print(f"\n===== {name} =====")

    for col in ["subject_id", "study_id", "dicom_id"]:
        if col in df.columns:
            print(f"{col}: {df[col].nunique():,}")

In [ ]:
# 심비대(Cardiomegaly)·폐부종(Edema)·흉수(Pleural Effusion)
chexpert['Cardiomegaly'].info()
chexpert['Edema'].info()
chexpert['Pleural Effusion'].info()


In [ ]:
# 주요 3개 소견별 레이블 분포 확인
# - 분석 대상: 심비대(Cardiomegaly), 폐부종(Edema), 흉수(Pleural Effusion)
# - 양성(1), 음성(0), 불확실(-1), 결측(NaN) 개수 확인
# - 결측값을 포함하여 각 소견의 레이블 분포 파악
target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

# 각 레이블의 양성, 음성, 불확실, 결측 분포
for label in target_labels:
    print(f"\n===== {label} =====")
    print(
        chexpert[label].value_counts(
            dropna=False
        )
    )

In [ ]:
# 주요 3개 소견별 레이블 분포표 생성
# - 분석 대상: 심비대, 폐부종, 흉수
# - 양성(1), 음성(0), 불확실(-1), 결측(NaN) 개수 집계
# - 전체 검사 수 대비 레이블별 비율(%) 계산
# - 소견별 개수와 비율을 하나의 DataFrame으로 정리

target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

label_names = {
    1.0: "Positive",
    0.0: "Negative",
    -1.0: "Uncertain",
    "NaN": "Missing"
}
results = []
for label in target_labels:
    counts = chexpert[label].value_counts(dropna=False)
    for value, name in label_names.items():
        if value == "NaN":
            count = chexpert[label].isna().sum()
        else:
            count = counts.get(value, 0)

        results.append({
            "Finding": label,
            "Label": name,
            "Count": count,
            "Percent": round(count / len(chexpert) * 100, 2)
        })
label_distribution= pd.DataFrame(results)
label_distribution

In [ ]:
label_distribution[
    label_distribution["Label"] == "Positive"
]
triple_positive = chexpert[
    (chexpert["Cardiomegaly"] == 1) &
    (chexpert["Edema"] == 1) &
    (chexpert["Pleural Effusion"] == 1)
].copy()

print("세 가지 소견 모두 양성인 검사 수:", len(triple_positive))

display(triple_positive[
    ["subject_id", "study_id",
     "Cardiomegaly", "Edema", "Pleural Effusion"]
].head(10))

In [ ]:
# 주요 3개 소견의 동시 양성 검사 추출
# - 심비대, 폐부종, 흉수 각각의 양성(1) 분포 확인
# - 세 소견이 모두 양성(1)인 검사 추출
# - 해당 검사 수 및 상위 10개 검사 정보 확인

total = len(chexpert)
triple_count = len(triple_positive)

print(f"전체 검사 수: {total:,}")
print(f"세 소견 모두 양성: {triple_count:,}")
print(f"전체 대비 비율: {triple_count / total * 100:.2f}%")

print(
    "고유 환자 수:",
    triple_positive["subject_id"].nunique()
)

In [ ]:
triple_positive.to_csv(
    "../results/tables/triple_positive.csv",
    index=False
)

In [ ]:
# 8가지 조합별 분포 
all_negative=chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 0)
]
Cardiomegaly_only = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 0)
]
Edema_only = chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] == 0)
]
Pleural_Effusion_only = chexpert[
    (chexpert['Cardiomegaly'] == 0) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] == 1)
]
Cardiomegaly_Edema = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] ==0)
]
Cardiomegaly_Pleural_Effusion=chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 0) &
    (chexpert['Pleural Effusion'] ==1)
]
Edema_Pleural_Effusion=chexpert[
    (chexpert['Cardiomegaly'] == 0) &
        (chexpert['Edema'] == 1) &
        (chexpert['Pleural Effusion'] ==1)

]
all_positive = chexpert[
    (chexpert['Cardiomegaly'] == 1) &
    (chexpert['Edema'] == 1) &
    (chexpert['Pleural Effusion'] == 1)
]

In [ ]:
# 8가지 조합별 분포표

combined_distribution = pd.DataFrame({
    'Combination': [
        'All Negative',
        'Cardiomegaly Only',
        'Edema Only',
        'Pleural Effusion Only',
        'Cardiomegaly + Edema',
        'Cardiomegaly + Pleural Effusion',
        'Edema + Pleural Effusion',
        'All Positive'
    ],
    'Count': [
        len(all_negative),
        len(Cardiomegaly_only),
        len(Edema_only),
        len(Pleural_Effusion_only),
        len(Cardiomegaly_Edema),
        len(Cardiomegaly_Pleural_Effusion),
        len(Edema_Pleural_Effusion),
        len(all_positive)
    ]
})

combined_distribution['Percent'] = (
    combined_distribution['Count'] / len(chexpert) * 100
).round(2)

display(combined_distribution)

In [ ]:
# ============================================================
# 결측·불확실 레이블에 따른 데이터 손실 분석
# - 3개 소견 모두 명확한 레이블(0, 1)을 가진 검사 수
# - 결측(NaN) 또는 불확실(-1)이 포함된 검사 수
# - 전체 대비 학습 후보 데이터 비율 확인
# ============================================================

target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

valid_mask = chexpert[target_labels].isin([0, 1]).all(axis=1)

total_count = len(chexpert)
valid_count = valid_mask.sum()
excluded_count = (~valid_mask).sum()

print(f"전체 검사 수: {total_count:,}")
print(f"명확한 레이블을 가진 검사 수: {valid_count:,}")
print(f"결측·불확실 포함 검사 수: {excluded_count:,}")
print(f"학습 후보 데이터 비율: {valid_count / total_count * 100:.2f}%")
print(f"제외 대상 비율: {excluded_count / total_count * 100:.2f}%")

In [ ]:
# ============================================================
# 09. 주요 3개 소견의 결측(NaN) 조합 분석
# - 검사별 NaN 개수 계산
# - NaN 0개, 1개, 2개, 3개인 검사 수 확인
# - 세 소견 모두 NaN인 검사 수 확인 (제외 후보)
# ============================================================

target_labels = [
    "Cardiomegaly",
    "Edema",
    "Pleural Effusion"
]

# 검사별 NaN 개수
nan_count = chexpert[target_labels].isna().sum(axis=1)

print("NaN 0개:", (nan_count == 0).sum())
print("NaN 1개:", (nan_count == 1).sum())
print("NaN 2개:", (nan_count == 2).sum())
print("NaN 3개 (모두 결측):", (nan_count == 3).sum())

print("NaN 1개 이상:", (nan_count >= 1).sum())

In [ ]:
# ============================================================
# 10. 소견별 NaN 조합 분석
# - 각 소견에서 단독으로 NaN인 검사 수
# - 두 소견이 동시에 NaN인 검사 수
# - 세 소견 모두 NaN인 검사 수
# ============================================================

c = chexpert["Cardiomegaly"].isna()
e = chexpert["Edema"].isna()
p = chexpert["Pleural Effusion"].isna()

nan_results = {
    "심비대만 NaN": (c & ~e & ~p).sum(),
    "폐부종만 NaN": (~c & e & ~p).sum(),
    "흉수만 NaN": (~c & ~e & p).sum(),
    "심비대 + 폐부종 NaN": (c & e & ~p).sum(),
    "심비대 + 흉수 NaN": (c & ~e & p).sum(),
    "폐부종 + 흉수 NaN": (~c & e & p).sum(),
    "세 소견 모두 NaN": (c & e & p).sum()
}

nan_distribution = pd.DataFrame(
    nan_results.items(),
    columns=["NaN 조합", "검사 수"]
)

display(nan_distribution)

## EDA 결과 요약

### 1. 분석 목적

MIMIC-CXR-JPG의 CheXpert, Metadata, Split CSV를 확인하고, 심비대(Cardiomegaly)·폐부종(Edema)·흉수(Pleural Effusion)의 레이블 분포, 복합 소견, 결측·불확실 현황을 파악하였다.

아래 결과는 원본 CSV를 기준으로 하며, 검사 단위 통계와 이미지 단위 통계를 구분하여 작성하였다.

### 2. 데이터 구조

| 데이터 | 행 수 | 열 수 | 행의 단위 |
|---|---:|---:|---|
| CheXpert | 227,827 | 16 | 검사 |
| Metadata | 377,110 | 12 | 이미지 |
| Split | 377,110 | 4 | 이미지 |

| 데이터 | 고유 환자 수 | 고유 검사 수 | 고유 이미지 수 |
|---|---:|---:|---:|
| CheXpert | 65,379 | 227,827 | 해당 컬럼 없음 |
| Metadata | 65,379 | 227,835 | 377,110 |
| Split | 65,379 | 227,835 | 377,110 |

- `subject_id`: 환자 식별자
- `study_id`: 검사 식별자
- `dicom_id`: 이미지 식별자
- CheXpert와 Metadata/Split의 고유 검사 수에 8건의 차이가 확인되었다.
- 세 소견 컬럼의 자료형은 모두 `float64`이다.

### 3. 촬영 방향 분포

전체 이미지 377,110장을 기준으로 촬영 방향을 확인하였다.

| 촬영 방향 | 이미지 수 |
|---|---:|
| AP | 147,173 |
| PA | 96,161 |
| LATERAL | 82,853 |
| LL | 35,133 |
| 결측(NaN) | 15,769 |
| 기타 방향 | 21 |
| **전체** | **377,110** |

AP와 PA의 합계는 243,334장이다. 원본 데이터에는 AP/PA 외의 촬영 방향과 촬영 방향 정보가 결측인 이미지도 포함되어 있었다.

### 4. 기존 Train / Validate / Test 분포

제공된 Split CSV에 기록된 기존 분할의 이미지 수와 비율을 확인하였다.

| 분할 | 이미지 수 | 전체 이미지 대비 비율 |
|---|---:|---:|
| Train | 368,960 | 97.84% |
| Validate | 2,991 | 0.79% |
| Test | 5,159 | 1.37% |

이 결과는 원본 데이터의 기존 분할 분포이다.

### 5. 소견별 레이블 분포

전체 검사 227,827건을 기준으로 양성(1), 음성(0), 불확실(-1), 결측(NaN)의 개수와 비율을 계산하였다.

| 소견 | 양성(1) | 음성(0) | 불확실(-1) | 결측(NaN) |
|---|---:|---:|---:|---:|
| 심비대 | 44,845 (19.68%) | 15,911 (6.98%) | 6,043 (2.65%) | 161,028 (70.68%) |
| 폐부종 | 27,018 (11.86%) | 25,641 (11.25%) | 13,174 (5.78%) | 161,994 (71.10%) |
| 흉수 | 54,300 (23.83%) | 27,158 (11.92%) | 5,814 (2.55%) | 140,555 (61.69%) |

- 양성 레이블 비율은 흉수에서 가장 높았다.
- 세 소견 모두 결측 비율이 60%를 넘었다.
- 불확실 레이블 비율은 폐부종에서 가장 높았다.

### 6. 복합 소견 분석

세 레이블이 모두 0 또는 1인 검사에 대해 8가지 조합을 분석하였다. 아래 비율의 분모는 전체 검사 227,827건이다.

| 소견 조합 | 검사 수 | 전체 검사 대비 비율 |
|---|---:|---:|
| 세 소견 모두 음성 | 3,319 | 1.46% |
| 심비대만 양성 | 4,506 | 1.98% |
| 폐부종만 양성 | 276 | 0.12% |
| 흉수만 양성 | 554 | 0.24% |
| 심비대 + 폐부종 양성 | 1,354 | 0.59% |
| 심비대 + 흉수 양성 | 2,557 | 1.12% |
| 폐부종 + 흉수 양성 | 862 | 0.38% |
| 세 소견 모두 양성 | 6,323 | 2.78% |

- 8가지 조합에 포함되는 검사는 총 19,751건이다. NaN 또는 -1이 포함된 검사는 이 표에 포함되지 않는다.
- ‘심비대만 양성’은 심비대=1, 폐부종=0, 흉수=0인 경우를 의미한다. 다른 단독 양성 및 두 소견 양성 조합도 나머지 소견이 0인 경우를 기준으로 한다.
- 세 소견 모두 양성인 검사는 6,323건이며, 해당 고유 환자 수는 3,543명이다.

### 7. 결측·불확실 분석

#### 7.1 세 레이블이 모두 0 또는 1인 검사

결측 또는 불확실 레이블이 포함된 검사를 모두 제외한다고 가정했을 때 남는 검사 수를 계산하였다.

| 구분 | 검사 수 | 전체 검사 대비 비율 |
|---|---:|---:|
| 세 레이블 모두 0 또는 1 | 19,751 | 8.67% |
| NaN 또는 -1이 하나 이상 포함 | 208,076 | 91.33% |
| **전체** | **227,827** | **100.00%** |

결측·불확실을 포함한 검사를 모두 제외하면 전체의 8.67%만 남는다. 이는 조건별 데이터 규모를 분석한 결과이다.

#### 7.2 검사별 NaN 개수

| 세 소견 중 NaN 개수 | 검사 수 |
|---|---:|
| 0개 | 25,836 |
| 1개 | 45,668 |
| 2개 | 51,060 |
| 3개 | 105,263 |
| NaN 1개 이상 | 201,991 |

NaN이 0개인 25,836건에는 불확실(-1)이 포함된 검사도 들어 있다. 따라서 세 레이블이 모두 0 또는 1인 19,751건과 구분해야 한다.

#### 7.3 소견별 NaN 조합

| NaN 조합 | 검사 수 |
|---|---:|
| 심비대만 NaN | 17,203 |
| 폐부종만 NaN | 18,641 |
| 흉수만 NaN | 9,824 |
| 심비대 + 폐부종 NaN | 25,592 |
| 심비대 + 흉수 NaN | 12,970 |
| 폐부종 + 흉수 NaN | 12,498 |
| 세 소견 모두 NaN | 105,263 |

각 조합은 서로 겹치지 않으며, 표에서 NaN으로 표시하지 않은 소견의 값은 0, 1 또는 -1일 수 있다.

### 8. 주요 분석 결과

- 검사 단위 레이블 데이터와 이미지 단위 메타데이터·분할 데이터의 구조를 확인하였다.
- 촬영 방향에는 AP, PA뿐 아니라 LATERAL, LL, 기타 방향 및 결측값이 포함되어 있었다.
- 세 소견 모두 결측 비율이 높았으며, 불확실 레이블도 존재하였다.
- 세 소견 동시 양성 검사는 6,323건으로, 전체 검사의 2.78%였다.
- 세 레이블이 모두 0 또는 1인 검사는 전체의 8.67%이므로, 결측·불확실 처리 기준에 따라 활용 가능한 데이터 규모가 크게 달라질 수 있다.